In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from IPython.display import Math
from matplotlib.colors import ListedColormap
from sklearn import datasets
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [ ]:
%matplotlib inline

In [ ]:
iris = datasets.load_iris()
lf_x = pl.LazyFrame(iris["data"], schema=iris["feature_names"])
# print(lf_x.collect())
print("target_names:", iris["target_names"])
lf_y = pl.LazyFrame(iris["target"], schema=["target"])
# print(lf_y.collect())
lf_iris = (
    pl.concat([lf_x, lf_y], how="horizontal")
    .select(["sepal length (cm)", "petal length (cm)", "target"])
    .filter(pl.col("target").is_in([0, 1]))
)
lf_iris.collect()

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    lf_iris.select(["sepal length (cm)", "petal length (cm)"]).collect(),
    lf_iris.select(["target"]).collect(),
    test_size=0.3,
    random_state=0,
)

In [ ]:
x_train

In [ ]:
x_test

In [ ]:
sc = StandardScaler()
sc.fit(x_train)
x_train_std = sc.transform(x_train)
x_test_std = sc.transform(x_test)

In [ ]:
x_train_std

In [ ]:
Math(r"z=w^Tx")

In [ ]:
Math(r"\phi{(z)}=\frac{1}{1+e^{-z}}")

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

In [ ]:
z = np.arange(-7, 7, 0.1)
phi_z = sigmoid(z)

plt.plot(z, phi_z)
plt.axvline(0.0, color="k")
plt.ylim(-0.1, 1.1)
plt.xlabel("z")
plt.ylabel("$\phi (z)$")

# y axis ticks and gridline
plt.yticks([0.0, 0.5, 1.0])
ax = plt.gca()
ax.yaxis.grid(True)

plt.tight_layout()
# plt.savefig('./figures/sigmoid.png', dpi=300)
plt.show()

In [ ]:
y_train.select("target").to_numpy().flatten()

In [ ]:
lr = LogisticRegression()
lr.fit(x_train_std, y_train.select("target").to_numpy().flatten())

In [ ]:
def plot_decision_regions(x, y, classifier, test_idx=None, resolution=0.02):
    # setup marker generator and color map
    markers = ("s", "x", "o", "^", "v")
    colors = ("red", "blue", "lightgreen", "gray", "cyan")
    cmap = ListedColormap(colors[: len(np.unique(y))])

    # plot the decision surface
    x1_min, x1_max = x[:, 0].min() - 1, x[:, 0].max() + 1
    x2_min, x2_max = x[:, 1].min() - 1, x[:, 1].max() + 1
    xx1, xx2 = np.meshgrid(
        np.arange(x1_min, x1_max, resolution), np.arange(x2_min, x2_max, resolution)
    )

    z = classifier.predict(np.array([xx1.ravel(), xx2.ravel()]).T)
    z = z.reshape(xx1.shape)
    plt.contourf(xx1, xx2, z, alpha=0.4, cmap=cmap)
    plt.xlim(xx1.min(), xx1.max())
    plt.ylim(xx2.min(), xx2.max())

    for idx, cl in enumerate(np.unique(y)):
        plt.scatter(
            x=x[y == cl, 0],
            y=x[y == cl, 1],
            alpha=0.6,
            c=cmap(idx),
            edgecolors="black",
            marker=markers[idx],
            label=cl,
        )

    # highlight test samples
    if test_idx:
        # plot all samples
        if not str(np.version.version) >= "1.9.0":
            x_test, y_test = x[list(test_idx), :], y[list(test_idx)]
            warnings.warn("Please update to NumPy 1.9.0 or newer")
        else:
            X_test, y_test = x[test_idx, :], y[test_idx]
        plt.scatter(
            x_test[:, 0],
            x_test[:, 1],
            alpha=1.0,
            c="",
            edgecolors="black",
            marker="o",
            label="test test",
            s=55,
        )

In [ ]:
plot_decision_regions(
    x_train_std, y_train.select("target").to_numpy().flatten(), classifier=lr
)
plt.xlabel("sepal length (cm) [standardized]")
plt.ylabel("petal width [standardized]")
plt.legend(loc="upper left")
plt.tight_layout()

In [ ]:
lr.predict(x_test_std)

In [ ]:
y_test.select("target").get_column("target").to_numpy()

In [ ]:
# 1. 先把預測結果與真實答案轉成 NumPy 陣列
y_pred = lr.predict(x_test_std)
y_true = y_test.select("target").get_column("target").to_numpy()

# 2. 計算錯誤數
error = np.sum(y_pred != y_true)
print(error)

In [ ]:
lr.predict_proba(x_test_std)

: 